# 04 · Adversarial attacks

Instead of degrading the whole video, optimise the **texture painted on the
object** until the network stops seeing the collision. The object still follows
a physically consistent trajectory; only its surface changes.

Three attacks, all on hit trajectories starting from a black texture:

- **PGD** — 100 gradient steps of size ε/10 inside an ε-ball around black; the
  step halves and rewinds to the best iterate every 20 steps. 2 random
  restarts, 15 trajectories, ε<sub>L∞</sub> ∈ [1, 6, 8, 10, 15] and
  ε<sub>L2</sub> ∈ [100, 250, 500, 1000, 2000].
- **CW** — Adam (lr 0.1, 150 steps) trading misclassification against how far
  the texture moved, c ∈ [1, 10, 10², 10³, 10⁴, 10⁵], 10 trajectories.
- **NES** — no gradients: 100 steps, each estimating the direction from 15
  random probes, σ ∈ [1, 3, 6, 9, 12], 15 trajectories.

Runs write snapshots and a loss history under `PGD/`, `CW/`, `NES/` (or
`$FLYNET_RESULTS`); the last cells turn those into `R` and `LERS`.

In [ ]:
import json, torch, numpy as np
from flynet_robustness import config
from flynet_robustness.models import load_network
from flynet_robustness.metrics import bootstrap_robustness, lers

device = 'cuda' if torch.cuda.is_available() else 'cpu'
network_id = 'FlyNet'   # any id: 'CTRNN', 'sCTRNN', 'vCNN_5_frames', 'RAFT_12', 'RAFT_12_S', ...

net, decoder = load_network(network_id, device=device)
trajectories = json.load(open('data/configs/attack_trajectories.json'))
attack_kwargs = [dict(config.DEFAULT_STIMULUS, **t, device=device, a=1,
                      mode='avg_pool_adaptive') for t in trajectories]

from flynet_robustness.attacks import run_cw, run_nes, run_pgd
from flynet_robustness.attacks.common import final_values

## PGD

For every network at once:

```bash
python scripts/run_attacks.py --networks all --attack pgd
```

In [ ]:
PGD_EPS = {'l2': [100, 250, 500, 1000, 2000], 'linf': [1, 6, 8, 10, 15]}
for norm, epss in PGD_EPS.items():
    for eps in epss:
        for i, kw in enumerate(attack_kwargs[:15]):
            torch.manual_seed(2 * (i + 1))
            run_pgd(net, decoder, network_id, norm=norm, eps=eps, lr=eps / 10, steps=100,
                    n_restarts=2, base_kwargs=kw, run_id=i + 1, device=device)

One run, to see what it leaves behind: the texture from the best restart
(`best_texture.pt`), per-restart snapshots, and the loss history.

In [ ]:
from flynet_robustness.plotting import plot_texture
out_dir = run_pgd(net, decoder, network_id, norm='l2', eps=1000, lr=100, steps=100,
                  n_restarts=2, base_kwargs=attack_kwargs[0], run_id=1, device=device)
print(out_dir, json.load(open(out_dir / 'best.json')))
plot_texture(torch.load(out_dir / 'best_texture.pt', weights_only=False));

## CW and NES

In [ ]:
CW_C, NES_SIGMA = [1, 10, 1e2, 1e3, 1e4, 1e5], [1, 3, 6, 9, 12]
for i, kw in enumerate(attack_kwargs[:10]):
    for c in CW_C:
        for norm in ('linf', 'l2'):
            torch.manual_seed(2 * (i + 1))
            run_cw(net, decoder, network_id, norm=norm, c=c, steps=150, lr=0.1,
                   base_kwargs=kw, run_id=i + 1, device=device)
for i, kw in enumerate(attack_kwargs[:15]):
    for sigma in NES_SIGMA:
        torch.manual_seed(2 * (i + 1))
        run_nes(net, decoder, network_id, sigma=sigma, steps=100, n_samples=15, lr=1.0,
                base_kwargs=kw, run_id=i + 1, device=device)

## Robustness and efficiency

For PGD and NES, how much accuracy survives the attack, taking the best
restart. For CW, how often the attack succeeded and how far it had to move the
texture.

In [ ]:
for name, root, norm, values, n in [('PGD L2', 'PGD', 'l2', PGD_EPS['l2'], 15),
                                     ('PGD Linf', 'PGD', 'linf', PGD_EPS['linf'], 15),
                                     ('NES', 'NES', 'gaussian', NES_SIGMA, 15)]:
    acc = 1 - final_values(root, 'Black', network_id, norm, values, n)
    R, R_std = bootstrap_robustness(values, acc)
    print(f'{name:9s} R = {R:.3f} ± {R_std:.3f}   LERS = {lers(R, network_id):.3f}')

for norm in ('linf', 'l2'):
    success = final_values('CW', 'Black', network_id, norm, CW_C, 10)
    size = np.where(success > 0,
                    final_values('CW', 'Black', network_id, norm, CW_C, 10, key='max_delta'),
                    np.nan)
    print(f'CW {norm:4s} success {np.nanmean(success, 1).round(3)}  size {np.nanmean(size, 1).round(3)}')

In [ ]:
import csv
for label, norm in [('PGD L2', 'l2'), ('PGD Linf', 'linf')]:
    ref = {r['network']: float(r['R']) for r in
           csv.DictReader(open(f'data/reference_results/PGD_{norm}.csv'))}
    acc = 1 - final_values('PGD', 'Black', network_id, norm, PGD_EPS[norm], 15)
    R, _ = bootstrap_robustness(PGD_EPS[norm], acc)
    print(f'{label:9s} ours {R:.3f}   published {ref[config.canonical(network_id)]:.3f}')

Restoring a fooled texture: notebook 07. Reusing one network's textures on the
others: notebook 08.